# SEC EDGAR 재무데이터 수집 (YTD → Quarterly 변환 수정 버전)

## 수정 사항
- YTD 누적 데이터를 분기별 데이터로 정확히 변환
- Revenue, Net Income 등 손익계산서 항목의 중복 값 문제 해결

In [5]:
import sys
import pandas as pd
import numpy as np
import time

# 경로만 추가 (모듈 import 전)
sys.path.insert(0, r"C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast\US_Market\collect")

# 직접 파일 경로로 import (패키지 우회)
import importlib.util

def load_module_from_path(module_name, file_path):
    """파일 경로에서 직접 모듈 로드"""
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

base_path = r"C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast\US_Market\collect"

# 각 모듈 개별 로드
get_us_ticker_mod = load_module_from_path(
    "get_us_ticker",
    f"{base_path}\\sec_data_pipeline\\collectors\\get_us_ticker.py"
)

rate_limiter_mod = load_module_from_path(
    "rate_limiter",
    f"{base_path}\\sec_data_pipeline\\collectors\\rate_limiter.py"
)

analyzer_mod = load_module_from_path(
    "integrated_financial_analyzer",
    f"{base_path}\\sec_data_pipeline\\valuation\\integrated_financial_analyzer_mysql_fixed.py"
)

db_mod = load_module_from_path(
    "db_manager",
    f"{base_path}\\sec_data_pipeline\\storage\\db_manager.py"
)

func_mod = load_module_from_path(
    "stock_invest_function",
    f"{base_path}\\DATA\\stock_invest_function.py"
)

# 클래스/함수 추출
get_filtered_us_tickers = get_us_ticker_mod.get_filtered_us_tickers
AdaptiveRateLimiter = rate_limiter_mod.AdaptiveRateLimiter
IntegratedFinancialAnalyzer = analyzer_mod.IntegratedFinancialAnalyzer
DBManager = db_mod.DBManager
get_db_host = func_mod.get_db_host

print("모든 모듈 로드 완료")

ModuleNotFoundError: No module named 'US_Market'

## YTD → Quarterly 변환 함수

In [ ]:
def convert_ytd_to_quarterly(df):
    """
    손익계산서 항목을 YTD(누적)에서 Quarterly(분기별)로 변환
    
    Parameters:
    -----------
    df : DataFrame
        날짜 인덱스를 가진 재무 데이터프레임
    
    Returns:
    --------
    DataFrame : 분기별 데이터로 변환된 데이터프레임
    """
    
    df = df.copy()
    df.index = pd.to_datetime(df.index)
    df = df.sort_index()
    
    # YTD 방식으로 제공되는 손익계산서 항목들
    ytd_columns = [
        'revenue', 'net_income', 'operating_income', 'gross_profit',
        'cost_of_revenue', 'operating_expenses', 'research_development',
        'selling_general_administrative', 'interest_expense', 
        'income_tax_expense', 'depreciation_amortization',
        'ebitda', 'ebit'
    ]
    
    # 회계연도와 분기 추가
    df['fiscal_year'] = df.index.year
    df['fiscal_quarter'] = df.index.quarter
    
    for col in ytd_columns:
        if col not in df.columns:
            continue
        
        if df[col].isna().all():
            continue
        
        quarterly_values = []
        
        for idx, row in df.iterrows():
            fiscal_year = row['fiscal_year']
            fiscal_quarter = row['fiscal_quarter']
            current_value = row[col]
            
            # NaN 처리
            if pd.isna(current_value):
                quarterly_values.append(np.nan)
                continue
            
            # Q1은 그대로 사용 (연초 누적 = 1분기 실적)
            if fiscal_quarter == 1:
                quarterly_values.append(current_value)
            else:
                # 같은 회계연도의 이전 분기 찾기
                prev_mask = (
                    (df['fiscal_year'] == fiscal_year) & 
                    (df['fiscal_quarter'] == fiscal_quarter - 1)
                )
                
                if prev_mask.any():
                    prev_ytd = df.loc[prev_mask, col].iloc[0]
                    
                    if pd.isna(prev_ytd):
                        # 이전 분기 데이터가 없으면 현재 값 사용
                        quarterly_values.append(current_value)
                    else:
                        # 현재 누적값 - 이전 누적값 = 당 분기 실적
                        quarterly_value = current_value - prev_ytd
                        quarterly_values.append(quarterly_value)
                else:
                    # 이전 분기 데이터가 없으면 현재 값 사용
                    quarterly_values.append(current_value)
        
        df[col] = quarterly_values
    
    # 임시 컬럼 제거
    df = df.drop(columns=['fiscal_year', 'fiscal_quarter'])
    
    return df

## 설정

In [ ]:
START_DATE = "2025-01-01"
MAX_TICKERS = 5000
OFFSET = 0

# DB 설정
db_info = {
    "host": get_db_host(),
    "port": 3307,
    "user": "stox7412",
    "password": "Apt106503!~",
    "database": "investar",
}

# 객체 초기화
db_manager = DBManager(db_info)
analyzer = IntegratedFinancialAnalyzer(db_info=db_info, wrds_conn_str="여기에_WRDS_주소")

rate_limiter = AdaptiveRateLimiter(
    max_calls=8,
    time_window=1.0,
    min_calls=3,
    backoff_factor=0.8,
)

headers = {"User-Agent": "Hoyoung Research <stox1224@gmail.com>"}

# 티커 리스트
ALL_TICKERS = get_filtered_us_tickers()
TICKER_LIST = ALL_TICKERS[OFFSET : OFFSET + MAX_TICKERS]

# 테스트용
TICKER_LIST = ['GOOG']

print(f"총 {len(TICKER_LIST)}개의 티커를 수집합니다. (기준일: {START_DATE})")

## 메인 실행

In [ ]:
for i, ticker in enumerate(TICKER_LIST, start=1):
    print(f"\n=== [{i}/{len(TICKER_LIST)}] {ticker} 처리 시작 ===")
    
    rate_limiter.wait_if_needed()
    
    try:
        result = analyzer.analyze(ticker, headers=headers, table_name="us_fundq")
        if result is None:
            continue
        
        final_df, cik, entity_name = result
        
        # YTD → Quarterly 변환
        print(f"  변환 전 Revenue (최근 4개): {final_df['revenue'].tail(4).values}")
        final_df = convert_ytd_to_quarterly(final_df)
        print(f"  변환 후 Revenue (최근 4개): {final_df['revenue'].tail(4).values}")
        
        # 날짜 필터링
        if not final_df.empty:
            final_df.index = pd.to_datetime(final_df.index)
            final_df = final_df[final_df.index >= pd.to_datetime(START_DATE)]
        
        if final_df.empty:
            print(f"⚠ {ticker}: {START_DATE} 이후의 새로운 데이터가 없어 저장을 건너뜁니다.")
            continue
        
        # DB 저장
        if final_df.index.name != "date":
            final_df.index.name = "date"
        
        db_manager.save_normalized_data(
            ticker=ticker,
            cik=cik,
            df=final_df,
            item_mapping=None
        )
        print(f"✓ {ticker} ({entity_name}) {len(final_df)}건 저장 완료")
        
        # 검증: 같은 회계연도에 중복 값이 있는지 체크
        df_check = final_df.copy()
        df_check['fiscal_year'] = df_check.index.year
        for year in df_check['fiscal_year'].unique():
            year_data = df_check[df_check['fiscal_year'] == year]['revenue']
            if len(year_data) > 1 and year_data.nunique() == 1:
                print(f"  ⚠ 경고: {year}년 모든 분기 Revenue가 동일 ({year_data.iloc[0]:,.0f})")
        
    except Exception as e:
        msg = str(e)
        if "429" in msg:
            print(f"⚠ {ticker}: SEC 429 감지 → 60초 대기")
            rate_limiter.on_rate_limit_error()
            time.sleep(60)
            continue
        print(f"✗ {ticker} 처리 중 오류: {e}")
        continue

print("\n=== 작업 완료 ===")

## 결과 확인

In [ ]:
# 변환된 데이터 확인
if 'final_df' in locals():
    print("\n최종 데이터 (Billions):")
    display_df = final_df[['revenue', 'net_income', 'operating_income']].copy()
    display_df = display_df / 1e9
    print(display_df)
    
    # 중복 체크
    print("\n중복 값 체크:")
    for col in ['revenue', 'net_income']:
        unique_count = final_df[col].nunique()
        total_count = len(final_df[col].dropna())
        print(f"{col}: {unique_count}개 고유값 / {total_count}개 전체")
        
        if unique_count < total_count:
            print(f"  ⚠ 중복 값 존재!")
        else:
            print(f"  ✓ 모두 고유값")